# NexusTrade — Risk/Reward Calculator
## Calculate R:R Ratios for Trade Setups

Bereken risk:reward ratio's om te bepalen of een trade het waard is.

In [ ]:
import pandas as pd
import numpy as np

def calculate_risk_reward(entry, stop_loss, target1, target2=None):
    """
    Calculate Risk:Reward ratios
    
    Args:
        entry: Entry price
        stop_loss: Stop loss price
        target1: First target price
        target2: Second target price (optional)
    
    Returns:
        Dictionary with R:R ratios and price distances
    """
    risk = abs(entry - stop_loss)
    reward1 = abs(target1 - entry)
    
    if risk == 0:
        return {"error": "Risk cannot be zero"}
    
    rr1 = reward1 / risk
    
    result = {
        "entry": entry,
        "stop_loss": stop_loss,
        "target1": target1,
        "risk_amount": round(risk, 4),
        "reward1_amount": round(reward1, 4),
        "risk_reward_1": round(rr1, 2),
        "risk_pct": round((risk / entry) * 100, 2),
        "reward1_pct": round((reward1 / entry) * 100, 2),
        "meets_minimum": "✅ YES" if rr1 >= 1.5 else "🔴 NO (need ≥1.5:1)"
    }
    
    if target2:
        reward2 = abs(target2 - entry)
        rr2 = reward2 / risk
        result.update({
            "target2": target2,
            "reward2_amount": round(reward2, 4),
            "risk_reward_2": round(rr2, 2),
            "reward2_pct": round((reward2 / entry) * 100, 2)
        })
    
    return result

## Example: DVLT Long Setup Analysis

In [ ]:
# VOORBEELD: DVLT Long Setup
entry = 0.70
stop = 0.59
target1 = 0.90
target2 = 1.05

result = calculate_risk_reward(entry, stop, target1, target2)

print("═══════════════════════════════════════════")
print("RISK/REWARD ANALYSIS")
print("═══════════════════════════════════════════")
print(f"Entry Price:      ${result['entry']}")
print(f"Stop Loss:        ${result['stop_loss']} ({result['risk_pct']}% risk)")
print(f"\nTarget 1:         ${result['target1']} ({result['reward1_pct']}% gain)")
print(f"Risk:Reward 1:    {result['risk_reward_1']}:1  {result['meets_minimum']}")
print(f"\nTarget 2:         ${result['target2']} ({result['reward2_pct']}% gain)")
print(f"Risk:Reward 2:    {result['risk_reward_2']}:1")
print(f"\nRisk Amount:      ${result['risk_amount']} per share")
print("═══════════════════════════════════════════")

# NexusTrade Rule Check
print("\n📋 NEXUSTRADE RULE CHECK:")
if result['risk_reward_1'] >= 1.5:
    print("✅ Meets minimum 1.5:1 R:R for Target 1")
else:
    print("🔴 Does NOT meet minimum 1.5:1 R:R — SKIP THIS TRADE")
    
if 'risk_reward_2' in result and result['risk_reward_2'] >= 3.0:
    print("✅ Meets preferred 3:1 R:R for Target 2")
elif 'risk_reward_2' in result:
    print(f"⚠️  Target 2 is {result['risk_reward_2']}:1 (prefer ≥3:1)")

## Target Optimization
Find the optimal target prices for your desired R:R:

In [ ]:
# Test verschillende targets om de beste te vinden
print("\nTARGET OPTIMIZATION:")
print("─" * 50)
targets = [0.75, 0.80, 0.85, 0.90, 0.95, 1.00, 1.05, 1.10, 1.15, 1.20]

for t in targets:
    r = calculate_risk_reward(entry, stop, t)
    if r['risk_reward_1'] >= 3.0:
        status = "✅✅ Excellent (≥3:1)"
    elif r['risk_reward_1'] >= 2.0:
        status = "✅ Good (≥2:1)"
    elif r['risk_reward_1'] >= 1.5:
        status = "⚠️  Acceptable (≥1.5:1)"
    else:
        status = "🔴 Skip (<1.5:1)"
    
    print(f"Target ${t:>5.2f}: {r['risk_reward_1']:>4.1f}:1 ({r['reward1_pct']:>+6.2f}%)  {status}")

## Interactive Calculator

In [ ]:
# ==== PAS DEZE WAARDES AAN (MU trade setup) ====
MY_ENTRY = 927.50
MY_STOP = 897.00
MY_TARGET1 = 983.00
MY_TARGET2 = 1020.00  # Optional, set to None if not used
# ================================

my_result = calculate_risk_reward(MY_ENTRY, MY_STOP, MY_TARGET1, MY_TARGET2)

print("\n🎯 YOUR TRADE R:R ANALYSIS:")
print(f"Entry: ${MY_ENTRY} | Stop: ${MY_STOP} ({my_result['risk_pct']}% risk)")
print(f"\nTarget 1: ${MY_TARGET1} — R:R {my_result['risk_reward_1']}:1")
if MY_TARGET2:
    print(f"Target 2: ${MY_TARGET2} — R:R {my_result['risk_reward_2']}:1")

print(f"\n{my_result['meets_minimum']}")

## Reverse Calculator: Find Required Target
Calculate what target price you need for a desired R:R:

In [ ]:
def find_target_for_rr(entry, stop_loss, desired_rr):
    """Calculate required target price for desired R:R"""
    risk = abs(entry - stop_loss)
    required_reward = risk * desired_rr
    
    # Assuming LONG trade (entry > stop)
    target = entry + required_reward
    
    return round(target, 4)

# Voor de DVLT setup
print("\nREQUIRED TARGETS FOR SPECIFIC R:R RATIOS:")
print("─" * 50)
desired_ratios = [1.5, 2.0, 2.5, 3.0, 4.0, 5.0]

for ratio in desired_ratios:
    target = find_target_for_rr(MY_ENTRY, MY_STOP, ratio)
    gain_pct = ((target - MY_ENTRY) / MY_ENTRY) * 100
    print(f"{ratio}:1 R:R requires target of ${target:>6.4f} (+{gain_pct:.2f}%)")

## Win Rate Required for Profitability
Calculate breakeven win rate for different R:R ratios:

In [ ]:
def breakeven_winrate(risk_reward_ratio):
    """Calculate required win rate to break even"""
    return round(100 / (1 + risk_reward_ratio), 2)

print("\nBREAKEVEN WIN RATES:")
print("─" * 50)
print("R:R Ratio | Required Win % | To Profit (Need >%)")
print("─" * 50)

for ratio in [1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0]:
    winrate = breakeven_winrate(ratio)
    print(f"  {ratio}:1    |     {winrate:>5.1f}%      |      >{winrate:.1f}%")

print("\n💡 Key Insight:")
print("Higher R:R = You can be wrong more often and still profit")
print("At 3:1 R:R, you only need to win 25% of trades to break even!")